# Speculative decoding: accept, reject and resample

A small draft model proposes $K$ tokens; the large target model scores all of them in one forward pass; a rejection rule decides how many to keep, so the output has **exactly** the target model's distribution. Write the verification step for a batch, with the randomness passed in as uniforms so the result is deterministic.

Write `speculative_accept(draft, q, p, u, r)`:

- `draft`: `(B, K)` int, the drafted tokens $x_1 \dots x_K$.
- `q`: `(B, K, V)`, the draft model's distribution at each drafted position ($q_i(x_i) > 0$).
- `p`: `(B, K + 1, V)`, the target model's distributions at the same positions plus one extra position after the last draft.
- `u`: `(B, K)` and `r`: `(B,)`, uniforms in $[0, 1)$.

Per row:

1. Draft $i$ is **acceptable** if `u[i] * q[i, x_i] < p[i, x_i]` (that is, $u_i < p_i(x_i) / q_i(x_i)$). The number accepted, $n$, is the length of the leading run of acceptable drafts ($0 \le n \le K$).
2. If $n < K$, sample a replacement from the residual $\text{res} = \max(p_n - q_n, 0)$. If $n = K$, sample a bonus token from $\text{res} = p_K$.
3. Sample by inverse CDF with `r`: the token is the **first** index $j$ with $\mathrm{cumsum}(\text{res})_j > r \cdot \sum \text{res}$ (clip to $V - 1$).
4. The output row has length $K + 1$: drafts $x_1 \dots x_n$, then the sampled token, then `-1` padding.

Return `(n, out)`: `(B,)` int32 and `(B, K + 1)` int32. Write the logic for one row and `jax.vmap` it; no Python loop over $K$.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def speculative_accept(draft, q, p, u, r):
    """Return (accepted count (B,), output tokens (B, K + 1) padded with -1)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1, 1]], "dtype": "int64"}, {"__nd__": [[[0.5, 0.5, 0.0], [0.2, 0.8, 0.0]]], "dtype": "float64"}, {"__nd__": [[[0.5, 0.5, 0.0], [0.6, 0.2, 0.2], [0.1, 0.1, 0.8]]], "dtype": "float64"}, {"__nd__": [[0.9, 0.5]], "dtype": "float64"}, {"__nd__": [0.3], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [1], "dtype": "int32"}, {"__nd__": [[1, 0, -1]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1, 0]], "dtype": "int64"}, {"__nd__": [[[0.5, 0.5, 0.0], [0.2, 0.8, 0.0]]], "dtype": "float64"}, {"__nd__": [[[0.5, 0.5, 0.0], [0.6, 0.2, 0.2], [0.1, 0.1, 0.8]]], "dtype": "float64"}, {"__nd__": [[0.1, 0.1]], "dtype": "float64"}, {"__nd__": [0.95], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [2], "dtype": "int32"}, {"__nd__": [[1, 0, 2]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[0, 2, 2, 0], [4, 5, 4, 2], [4, 4, 4, 5]], "dtype": "int64"}, {"__nd__": [[[0.1901062254904881, 0.07270351209391873, 0.22999279306815842, 0.1331306708033117, 0.026530964190435102, 0.34753583435368796], [0.080979221594952, 0.16598061282254595, 0.5605401861363261, 0.11680416832714365, 0.03049613972424544, 0.04519967139478679], [0.38101702020497497, 0.032112633613535106, 0.17939804319677405, 0.2005938601111622, 0.04362862066661388, 0.16324982220693982], [0.33319903630060943, 0.08821062857534882, 0.1181587193571948, 0.32907719392311213, 0.05212376903073664, 0.07923065281299828]], [[0.21404988318543855, 0.08983394390353006, 0.02000893067277673, 0.08340557576818662, 0.5745423657668989, 0.018159300703169234], [0.18439595004039108, 0.17414875613077152, 0.06833276460931277, 0.14937449877768175, 0.04155730476874062, 0.38219072567310214], [0.5637992453685615, 0.041868081924486024, 0.08152262329451122, 0.11517096123205331, 0.05400983408609295, 0.14362925409429508], [0.044537968662808465, 0.2239250453756542, 0.09306184384660601, 0.11971777890448664, 0.20135145271130617, 0.31740591049913847]], [[0.034008232002572227, 0.20025922773584767, 0.2190664362612212, 0.08105516120043202, 0.42327235448615486, 0.04233858831377209], [0.043324967835556255, 0.10953114917829068, 0.037857979749931214, 0.3009212104323659, 0.09111890062241318, 0.41724579218144264], [0.042156067399016665, 0.04680848033203329, 0.46000493790287234, 0.042412391446623975, 0.35465432770327726, 0.05396379521617643], [0.13708755129234795, 0.3374117668547913, 0.09911961225501499, 0.03341193281423088, 0.06461814766109589, 0.3283509891225191]]], "dtype": "float64"}, {"__nd__": [[[0.17692182649843916, 0.29583706284948086, 0.15903305897026346, 0.19450846167524713, 0.12079409743936424, 0.05290549256720515], [0.12647126203706296, 0.08816884180692865, 0.010121899997555913, 0.07388608086292997, 0.1941687926369547, 0.5071831226585678], [0.03474705344303922, 0.15084473701762544, 0.23530935902824007, 0.28742914723778834, 0.06705965457473427, 0.22461004869857257], [0.11780902496765276, 0.17088516497468687, 0.10491434236700048, 0.24326844989196322, 0.12116925222709554, 0.2419537655716012], [0.08355856265555904, 0.2273269833914522, 0.4891391871158759, 0.08407988219347864, 0.043659073538773754, 0.07223631110486034]], [[0.10188938637525016, 0.0330574695878957, 0.1780804681316435, 0.10632685078716704, 0.4279214389917153, 0.1527243861263283], [0.34931616588695286, 0.29784262210466816, 0.08401005158307423, 0.09944171062638339, 0.08168218117735884, 0.08770726862156254], [0.2609681100912481, 0.06831078446954318, 0.07538789615179725, 0.05125669269446442, 0.46426916893100906, 0.07980734766193817], [0.23635115884634925, 0.11597786912149707, 0.34626782770846, 0.12534105504236334, 0.14039470061487036, 0.035667388666459986], [0.03180228101725651, 0.19795357968156607, 0.34984618853910265, 0.031015949460982562, 0.19247200132138045, 0.19690999997971176]], [[0.7679008919131514, 0.013567692651545848, 0.018720107187366657, 0.012787201253562626, 0.08320289735821566, 0.10382120963615792], [0.045717773660315666, 0.2727839452387715, 0.4902054917750635, 0.03765262551972446, 0.10413002066303438, 0.04951014314309055], [0.09849169895873808, 0.11693530539325597, 0.14689709011548138, 0.05154185182498006, 0.17121323254295837, 0.41492082116458606], [0.06190555407676924, 0.09426072217837025, 0.06406738737882761, 0.23253536766441468, 0.14355544391857422, 0.40367552478304397], [0.08436848864864865, 0.026623268028013958, 0.20332260350096779, 0.08583447158596776, 0.36272744655627154, 0.23712372168013016]]], "dtype": "float64"}, {"__nd__": [[0.35632481152250217, 0.6027091208654811, 0.8527781398917087, 0.09934704588744436], [0.9130483497021835, 0.2815465100019622, 0.7528672830533427, 0.18306745259063095], [0.9811005170515041, 0.5539529425787502, 0.04980512962203598, 0.3989290297105479]], "dtype": "float64"}, {"__nd__": [0.6362798562446526, 0.6876234997174707, 0.3358611188475825], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [1, 0, 0], "dtype": "int32"}, {"__nd__": [[0, 5, -1, -1, -1], [5, -1, -1, -1, -1], [0, -1, -1, -1, -1]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1, 3, 3], [0, 2, 1]], "dtype": "int64"}, {"__nd__": [[[0.04629601241860559, 0.318530570507893, 0.07968745751918911, 0.45419189636652896, 0.10129406318778329], [0.23737274042925477, 0.27506337623496374, 0.26543576121892276, 0.17985893355973753, 0.04226918855712126], [0.08823596846663352, 0.05779543630087002, 0.16781239828503536, 0.6455504305130159, 0.040605766434445335]], [[0.6838657599658138, 0.034346637926548296, 0.018843390928484843, 0.08510141318733415, 0.1778427979918191], [0.07080594747681927, 0.0364948545848896, 0.5849191557121349, 0.0337082312850311, 0.274071810941125], [0.3262535853963596, 0.19303131052794775, 0.29403990872791236, 0.09447337992509784, 0.09220181542268237]]], "dtype": "float64"}, {"__nd__": [[[0.04629601241860559, 0.318530570507893, 0.07968745751918911, 0.45419189636652896, 0.10129406318778329], [0.23737274042925477, 0.27506337623496374, 0.26543576121892276, 0.17985893355973753, 0.04226918855712126], [0.08823596846663352, 0.05779543630087002, 0.16781239828503536, 0.6455504305130159, 0.040605766434445335], [0.10889857692526221, 0.028371176695180026, 0.03709917045138709, 0.3414113568208916, 0.4842197191072791]], [[0.6838657599658138, 0.034346637926548296, 0.018843390928484843, 0.08510141318733415, 0.1778427979918191], [0.07080594747681927, 0.0364948545848896, 0.5849191557121349, 0.0337082312850311, 0.274071810941125], [0.3262535853963596, 0.19303131052794775, 0.29403990872791236, 0.09447337992509784, 0.09220181542268237], [0.38676377930630235, 0.31285119063084016, 0.09933614425885477, 0.1399665951584539, 0.061082290645548945]]], "dtype": "float64"}, {"__nd__": [[0.9313386501643816, 0.04830200303642118, 0.3988313699061712], [0.864488137494096, 0.8779171760747408, 0.03968037501368271]], "dtype": "float64"}, {"__nd__": [0.3256950035748125, 0.04247693407438702], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [3, 3], "dtype": "int32"}, {"__nd__": [[1, 3, 3, 3], [0, 2, 1, 0]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[4, 1, 4, 3, 5], [5, 4, 1, 4, 7], [0, 1, 4, 3, 6], [7, 4, 1, 6, 3]], "dtype": "int64"}, {"__nd__": [[[0.30253654812001823, 0.04670866154860472, 0.17212023992122136, 0.029491463532535105, 0.14205423790908245, 0.025343748906406383, 0.16943149103853147, 0.1123136090236003], [0.032233923595597146, 0.1419031688275636, 0.036963734110439656, 0.10780286382086193, 0.03976948347543087, 0.16937978035391812, 0.056683474708401176, 0.41526357110778744], [0.1119339838257185, 0.12166715873105971, 0.018831336951941793, 0.008870967607068675, 0.03216659120687601, 0.4361031577038031, 0.0279229862616798, 0.2425038177118526], [0.3010342243218503, 0.047883615169751086, 0.06950228945783134, 0.317176344917318, 0.06649270901659642, 0.05722320476938843, 0.04831679661613424, 0.09237081573113008], [0.037774405255451754, 0.14478948366112962, 0.05342869157706726, 0.13709619346537286, 0.03792106628391174, 0.5148947307017824, 0.037585674797872, 0.036509754257412416]], [[0.08432390294702122, 0.03464672090776345, 0.07879674771821248, 0.2316506532025892, 0.06964960565638341, 0.4287479849146542, 0.03689521117004718, 0.03528917348332881], [0.0887772124476618, 0.09589121509235846, 0.021223808918294822, 0.19288068586725382, 0.2612056015026655, 0.09413586134203207, 0.01821691205173424, 0.22766870277799917], [0.054295421635363085, 0.060660623420079524, 0.11711285795084779, 0.03614625982766981, 0.326525867469068, 0.03895244923083043, 0.20967766814327274, 0.1566288523228686], [0.40113288871386504, 0.11360578980550802, 0.16258499350754077, 0.0466842616251279, 0.05473252361477021, 0.0378135059299139, 0.14709866528127513, 0.03634737152199896], [0.3726314844828451, 0.07431967641496695, 0.025517054731525587, 0.04288785198608601, 0.26488086772187014, 0.05167156065552577, 0.150639443535761, 0.017452060471419447]], [[0.14491154898651673, 0.20034480596532908, 0.15416929223796044, 0.20411861755173114, 0.03518464143608569, 0.018143562571793025, 0.02557008440365179, 0.21755744684693215], [0.16036729642305478, 0.23958505377905034, 0.06192535351411801, 0.24104714434633093, 0.023415684555461044, 0.16355737907284784, 0.038854749259972204, 0.0712473390491648], [0.1328462876553399, 0.16545939159378723, 0.05940144945020268, 0.04895196466963243, 0.26721207008537706, 0.19626280651136038, 0.03583493189540223, 0.09403109813889805], [0.06338788964278433, 0.2046877389053355, 0.1235022075377982, 0.12094722495057068, 0.08655637360398029, 0.04636136178463859, 0.23536661992137878, 0.11919058365351351], [0.018973074142688416, 0.12935878666173295, 0.24560004857540244, 0.09057402351171026, 0.0755300015974274, 0.10038279536511636, 0.26645581413935837, 0.07312545600656377]], [[0.0799427603567674, 0.21378266949936628, 0.14241546454054124, 0.14777835807234757, 0.04509227071795458, 0.18029938516823094, 0.05688568757247373, 0.13380340407231808], [0.030482774005065966, 0.32350905132457414, 0.05253249695505721, 0.09644576911740424, 0.2660839399455299, 0.09335822314038655, 0.058397160011600696, 0.0791905855003812], [0.21855194612362588, 0.22161999006320207, 0.1423107558234391, 0.03934415250572833, 0.04898977403478533, 0.1403560246462887, 0.15293314536589045, 0.03589421143703998], [0.03505429457687398, 0.13984326364347308, 0.20666641920108392, 0.07595539749328246, 0.021478145529901505, 0.3381435360024465, 0.07245098698226819, 0.1104079565706704], [0.02801422365579717, 0.11492211414108042, 0.19407115817416826, 0.3173851952166272, 0.09597370018084216, 0.05008518288812017, 0.04464726685875723, 0.15490115888460743]]], "dtype": "float64"}, {"__nd__": [[[0.13732214717492808, 0.13120086354548638, 0.018405533948670883, 0.05018430733029212, 0.0914681584200614, 0.2687101137775688, 0.17367391924618023, 0.12903495655681219], [0.02384989937415491, 0.3677440836646031, 0.11467378796593555, 0.15485028947860105, 0.06432933296910551, 0.10205212790619897, 0.08917809879797912, 0.08332237984342179], [0.03764759221477258, 0.17573733877514947, 0.22027693938005719, 0.05298909719453513, 0.03965225988335315, 0.04006590969118522, 0.06674900576215625, 0.36688185709879106], [0.05679533011683005, 0.39380591929994196, 0.018659011905597416, 0.0221510501055015, 0.029944454630713633, 0.027767292736937274, 0.0895529075646776, 0.36132403363980053], [0.01934396679986021, 0.05600886996710227, 0.14805968059869562, 0.4276506162228339, 0.049364832301940276, 0.10618986131614554, 0.148928628879565, 0.04445354391385721], [0.1466487200771355, 0.059301366577766586, 0.03506412066204559, 0.04685781395607474, 0.262670390629871, 0.3161057473713219, 0.10800808505936794, 0.025343755666416636]], [[0.20646086602792227, 0.0613366781937436, 0.26925253574354546, 0.06804878794348854, 0.01980233809235554, 0.04665623381999837, 0.03201129809607294, 0.2964312620828734], [0.011201735263186182, 0.05879060261559984, 0.07736692587526638, 0.014065433817297869, 0.16062773496581162, 0.19367849492088635, 0.24399647791824364, 0.24027259462370829], [0.09445704160216561, 0.26319874615233835, 0.06700464344729647, 0.1072132486694652, 0.12929011881179192, 0.11182364372292027, 0.17253196442914603, 0.0544805931648761], [0.05194047338611146, 0.03283161768333504, 0.06269156956389411, 0.0797095237343879, 0.027931979200603335, 0.24458080582428313, 0.3982653493166156, 0.10204868129076951], [0.10913888836483318, 0.05376648856332102, 0.08421450677882068, 0.08057912435550268, 0.19760959734288022, 0.09931848859894114, 0.2003782130640355, 0.17499469293166556], [0.07719009065792315, 0.03359339050894707, 0.03888222980372691, 0.04354638923024563, 0.12120123987695917, 0.3987064349924357, 0.05013027295761363, 0.2367499519721488]], [[0.03529199126632082, 0.14548144664134313, 0.384456574210512, 0.12011686998445735, 0.05888049076012563, 0.06958867371666345, 0.15427238297793885, 0.03191157044263871], [0.03390771173698723, 0.0864551708937891, 0.35688649795954186, 0.036514250856237974, 0.21978732254278704, 0.17397219388348503, 0.058897154541938945, 0.03357969758523285], [0.012320466646891267, 0.01960079396308843, 0.07222452790853594, 0.12654524747654486, 0.03279922928520299, 0.034632057963244624, 0.028322390492396512, 0.6735552862640953], [0.01935403562372302, 0.01742025736950345, 0.2133747808335544, 0.02519957828503136, 0.1996766211906842, 0.03255610128723505, 0.20411188728699467, 0.28830673812327384], [0.07518209397331924, 0.11604858437703769, 0.04436219700790857, 0.28165851222028837, 0.08629873840892444, 0.045999974947017575, 0.18146356345142903, 0.16898633561407503], [0.11574750988523388, 0.07898469877493637, 0.3132166795431953, 0.06312493678140056, 0.051301976835655086, 0.09436040646352753, 0.20150479968465074, 0.08175899203140058]], [[0.0491588960350618, 0.03323735069301213, 0.041479397570537715, 0.04717608929911996, 0.6586545577640869, 0.015500626318902842, 0.057738304296759885, 0.09705477802251879], [0.031967748571761025, 0.02355098321929285, 0.13369273815217184, 0.3035693319835945, 0.23054926717030166, 0.0783527753095578, 0.04961429388897214, 0.1487028617043482], [0.020204311135356433, 0.04516163527916108, 0.4489271798840297, 0.15995388077980194, 0.12265154228054886, 0.08661051333971681, 0.07142657544916986, 0.04506436185221533], [0.10753605121348978, 0.10021710672588698, 0.10166309102650542, 0.005321279123144425, 0.2176812813258522, 0.14277180120276753, 0.2901454153383044, 0.03466397404404944], [0.007875799290770289, 0.018773603226357873, 0.02841966258134791, 0.5133448195493209, 0.03253232602223083, 0.04337227328859825, 0.1868430917539733, 0.1688384242874007], [0.19228831742183822, 0.045831182695098054, 0.4726575663053055, 0.10908650567758515, 0.031324699713637986, 0.10992361284891079, 0.017817192692065484, 0.021070922645558855]]], "dtype": "float64"}, {"__nd__": [[0.27047236654017326, 0.8309399031841385, 0.3928663361763871, 0.9354779265648844, 0.44847841223551854], [0.8687276382496664, 0.32883947714461537, 0.586911325682787, 0.717529489844935, 0.15601680986885036], [0.24700347185716853, 0.45120140489503247, 0.47464747834164045, 0.4293658783694102, 0.3142717428258056], [0.73791994239585, 0.33124013703987354, 0.9396631883745225, 0.4774560654321274, 0.07568162036949988]], "dtype": "float64"}, {"__nd__": [0.789611316391311, 0.08382772778639336, 0.046415368192409656, 0.7483565825167688], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [3, 0, 0, 0], "dtype": "int32"}, {"__nd__": [[4, 1, 4, 7, -1, -1], [0, -1, -1, -1, -1, -1], [2, -1, -1, -1, -1, -1], [4, -1, -1, -1, -1, -1]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[0, 2]], "dtype": "int64"}, {"__nd__": [[[0.9, 0.05, 0.05], [0.1, 0.1, 0.8]]], "dtype": "float64"}, {"__nd__": [[[0.05, 0.9, 0.05], [0.3, 0.3, 0.4], [1.0, 0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[0.5, 0.0]], "dtype": "float64"}, {"__nd__": [0.99], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0], "dtype": "int32"}, {"__nd__": [[1, -1, -1]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1, 1], [3, 1], [3, 0], [3, 3], [1, 3], [3, 0]], "dtype": "int64"}, {"__nd__": [[[0.15884687964104777, 0.6985663473809074, 0.11276906440240457, 0.029817708575640246], [0.12125597693115686, 0.4141332869155235, 0.3164441784756329, 0.1481665576776868]], [[0.07347843799850698, 0.014865357062334767, 0.11747928265108012, 0.794176922288078], [0.09184067294754292, 0.7669102922695843, 0.04328348078733167, 0.09796555399554104]], [[0.1252509931335076, 0.19619699449574832, 0.0330041601216308, 0.6455478522491134], [0.18857098791337076, 0.02976408041374578, 0.4876755582188381, 0.2939893734540453]], [[0.05117760783269555, 0.1630890883867381, 0.4138543755950588, 0.37187892818550755], [0.20066100432519385, 0.07397996663304038, 0.19357678063242503, 0.5317822484093407]], [[0.5589719028236172, 0.12075541189184591, 0.14709675947604675, 0.1731759258084901], [0.3340969533335857, 0.07514417453361939, 0.24330711996730764, 0.3474517521654873]], [[0.3944192030671118, 0.30364916801644193, 0.14017917197257374, 0.16175245694387236], [0.44827557573401644, 0.22842435468531996, 0.230867711759258, 0.09243235782140559]]], "dtype": "float64"}, {"__nd__": [[[0.23407179561589006, 0.36191432031165344, 0.1656050314856309, 0.23840885258682562], [0.4697848994415443, 0.14230458753899303, 0.2202613122441417, 0.16764920077532097], [0.5134966978091836, 0.10669165196093026, 0.3363424514047431, 0.043469198825143066]], [[0.639911377295396, 0.19695884864034763, 0.051703489487726566, 0.11142628457652994], [0.7354630459634872, 0.12486540673307593, 0.02625464488748955, 0.11341690241594728], [0.16278554226823683, 0.4509836025940632, 0.32897609952916296, 0.05725475560853704]], [[0.13065232081259068, 0.27740118083783677, 0.43703839382376314, 0.1549081045258095], [0.3602028343306337, 0.2418589117113831, 0.12466451361205298, 0.2732737403459303], [0.009177884272184808, 0.3982521936895298, 0.48846226127292647, 0.10410766076535885]], [[0.5073336607191523, 0.11628927775268964, 0.1891121080528839, 0.18726495347527422], [0.16046366516052343, 0.3666765038953408, 0.07777850345663108, 0.39508132748750474], [0.2898991891408563, 0.3532654897950164, 0.05224981934221555, 0.30458550172191173]], [[0.45975305321303517, 0.08553894503030078, 0.1645730662573485, 0.29013493549931557], [0.2412978491748248, 0.472462569839279, 0.1220762838320939, 0.16416329715380232], [0.43723212485275276, 0.17820727918186513, 0.23629461762633092, 0.14826597833905114]], [[0.2592397427716414, 0.2747468897027316, 0.059810553270968216, 0.4062028142546588], [0.1128775849417078, 0.30604976735434347, 0.16027512520291046, 0.4207975225010384], [0.25721709589637315, 0.34491568750889084, 0.22401188756814225, 0.1738553290265938]]], "dtype": "float64"}, {"__nd__": [[0.7724789739446739, 0.8721202925728574], [0.8955438711069994, 0.0028393175692975614], [0.4701648490729643, 0.6117869687582983], [0.016026323574327983, 0.8890437494958673], [0.05587235502778398, 0.09507866077046356], [0.47269618626949894, 0.008535884895274237]], "dtype": "float64"}, {"__nd__": [0.6881163603168442, 0.9927917279563652, 0.4778640199359204, 0.5401518038697205, 0.4039803199401625, 0.8587118756339828], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0, 0, 0, 1, 2, 2], "dtype": "int32"}, {"__nd__": [[3, -1, -1], [1, -1, -1], [2, -1, -1], [3, 1, -1], [1, 3, 0], [3, 0, 3]], "dtype": "int32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(speculative_accept, json.loads(_TESTS), 'jax', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")